# matvec — worked example 3: Solve the Normal Equations with Matvec for Least Squares

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matvec`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The normal equations for least squares are `(A.T @ A) x = A.T @ b`. Once `A.T @ A` is computed, solving the system via `torch.linalg.solve` reduces to a matvec application at prediction time: `A @ x_hat` predicts the target vector using the solved weights `x_hat: (N,)` and the matrix `A: (M, N)`.

## Worked solution

**Step 1 — create an overdetermined system.**
We generate `A: (5, 3)` (more equations than unknowns) and target `b: (5,)`.

**Step 2 — form the normal equations.**
We compute `ATA = A.T @ A: (3, 3)` and `ATb = A.T @ b: (3,)`.

**Step 3 — solve for x_hat.**
We call `torch.linalg.solve(ATA, ATb)` to get `x_hat: (3,)` — the least-squares solution.

**Step 4 — compute the fitted values as a matvec.**
The predictions are `y_hat = A @ x_hat: (5, 3) @ (3,) → (5,)`. We verify the shape is `(5,)` and compute the residual norm.

In [ ]:
import torch as t

t.manual_seed(11)
A = t.randn(5, 3)   # overdetermined: 5 equations, 3 unknowns
x_true = t.tensor([1.0, -0.5, 2.0])
b = A @ x_true + 0.1 * t.randn(5)   # noisy observations

# Normal equations: (A.T A) x = A.T b
ATA = A.T @ A   # (3,3)
ATb = A.T @ b   # (3,)

# Solve for least-squares solution
x_hat = t.linalg.solve(ATA, ATb)   # (3,)

# Fitted values as matvec: A @ x_hat -> (5,)
y_hat = A @ x_hat

print(f"A:     {A.shape}")
print(f"x_true: {x_true}")
print(f"x_hat:  {x_hat.round(decimals=4)}")
print(f"y_hat shape: {y_hat.shape}  (must be (5,), not (5,1))")
print(f"Residual |b - y_hat|: {(b - y_hat).norm():.4f}")
print(f"Recovered x_true well: {t.allclose(x_hat, x_true, atol=0.5)}")